## Step 2 — CWRU Data Preprocessing
**Dissertation: TinyML-Based Anomaly Detection**  
**Author: Thondupu Dileep | 2024AB05233**

Loads raw .mat vibration files → windows → features → saves .npz

In [ ]:
import numpy as np
import pandas as pd
import scipy.io as sio
from scipy.fft import rfft
from scipy.stats import kurtosis, skew
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

CWRU_DIR    = '../CWRU/raw'
OUT_DIR     = '../data/processed'
WINDOW_SIZE = 2048
OVERLAP     = 0.5
SR          = 48000

os.makedirs(OUT_DIR, exist_ok=True)
print('Ready')

### 3.1 — Load raw signal from .mat file

In [ ]:
FILE_MAP = {
    'Time_Normal_1_098.mat': (0, 'Normal'),
    'B007_1_123.mat'       : (1, 'Ball_007'),
    'B014_1_190.mat'       : (2, 'Ball_014'),
    'B021_1_227.mat'       : (3, 'Ball_021'),
    'IR007_1_110.mat'      : (4, 'InnerRace_007'),
    'IR014_1_175.mat'      : (5, 'InnerRace_014'),
    'IR021_1_214.mat'      : (6, 'InnerRace_021'),
    'OR007_6_1_136.mat'    : (7, 'OuterRace_007'),
    'OR014_6_1_202.mat'    : (8, 'OuterRace_014'),
    'OR021_6_1_239.mat'    : (9, 'OuterRace_021'),
}

def load_signal(filepath):
    mat = sio.loadmat(filepath)
    for k in mat.keys():
        if 'DE_time' in k:
            return mat[k].flatten().astype(np.float32)
    for k in mat.keys():
        if 'time' in k.lower() and not k.startswith('__'):
            return mat[k].flatten().astype(np.float32)
    raise ValueError(f'No signal in {filepath}')

# test on normal file
sig = load_signal(os.path.join(CWRU_DIR, 'Time_Normal_1_098.mat'))
print(f'Signal shape: {sig.shape}')
print(f'Duration: {len(sig)/SR:.1f}s  Min:{sig.min():.4f}  Max:{sig.max():.4f}')

### 3.2 — Windowing

In [ ]:
def make_windows(signal, win=2048, overlap=0.5):
    step = int(win * (1 - overlap))
    out  = []
    i    = 0
    while i + win <= len(signal):
        out.append(signal[i:i+win])
        i += step
    return np.array(out, dtype=np.float32)

# demo
wins = make_windows(sig)
print(f'Windows shape: {wins.shape}  ({wins.shape[0]} windows of {wins.shape[1]} samples)')
print(f'Each window = {WINDOW_SIZE/SR*1000:.1f} ms')

# visualise a few windows
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for i, ax in enumerate(axes):
    ax.plot(wins[i*50], linewidth=0.6)
    ax.set_title(f'Window {i*50}')
    ax.set_xlabel('Sample')
    ax.grid(True, alpha=0.3)
plt.suptitle('Sample Windows from Normal Signal')
plt.tight_layout()
plt.show()

### 3.3 — Feature Extraction (Time + Frequency domain)

In [ ]:
def extract_features(window):
    rms   = np.sqrt(np.mean(window**2))
    peak  = np.max(np.abs(window))
    stats = np.array([
        np.mean(window),
        np.std(window),
        rms,
        peak,
        peak / (rms + 1e-9),   # crest factor
        kurtosis(window),
        skew(window)
    ], dtype=np.float32)

    # first 256 FFT bins
    fft_mag = (np.abs(rfft(window))[:256] / len(window)).astype(np.float32)
    return np.concatenate([stats, fft_mag])  # 263 features

# test
feat = extract_features(wins[0])
print(f'Feature vector length: {len(feat)}')
print(f'Time features (7): mean={feat[0]:.4f}, std={feat[1]:.4f}, rms={feat[2]:.4f}')
print(f'FFT features (256): first 5 = {feat[7:12]}')

In [ ]:
# visualise feature importance — compare normal vs fault
sig_fault = load_signal(os.path.join(CWRU_DIR, 'IR021_1_214.mat'))
wins_f    = make_windows(sig_fault)

stats_n = np.array([extract_features(w)[:7] for w in wins[:200]])
stats_f = np.array([extract_features(w)[:7] for w in wins_f[:200]])

feat_names = ['Mean', 'Std', 'RMS', 'Peak', 'Crest', 'Kurtosis', 'Skewness']
fig, axes  = plt.subplots(2, 4, figsize=(14, 5))
axes = axes.flatten()

for i, name in enumerate(feat_names):
    axes[i].hist(stats_n[:, i], bins=30, alpha=0.6, color='green', label='Normal',  density=True)
    axes[i].hist(stats_f[:, i], bins=30, alpha=0.6, color='red',   label='IR021"',  density=True)
    axes[i].set_title(name, fontsize=9)
    axes[i].legend(fontsize=7)
    axes[i].grid(True, alpha=0.3)

axes[-1].axis('off')
plt.suptitle('Feature Distributions — Normal vs Inner Race Fault')
plt.tight_layout()
plt.show()

### 3.4 — Process all files

In [ ]:
all_raw   = []
all_feat  = []
all_label = []
all_name  = []

for fname, (label, name) in FILE_MAP.items():
    fpath = os.path.join(CWRU_DIR, fname)
    if not os.path.exists(fpath):
        print(f'  SKIP: {fname}')
        continue

    sig  = load_signal(fpath)
    wins = make_windows(sig, WINDOW_SIZE, OVERLAP)
    feat = np.array([extract_features(w) for w in wins], dtype=np.float32)

    all_raw.append(wins)
    all_feat.append(feat)
    all_label.extend([label] * len(wins))
    all_name.extend([name] * len(wins))

    print(f'  {name:<20} windows: {len(wins)}')

X_raw   = np.vstack(all_raw)
X_feat  = np.vstack(all_feat)
y       = np.array(all_label, dtype=np.int32)
y_names = np.array(all_name)

print(f'\nX_raw  shape : {X_raw.shape}')
print(f'X_feat shape : {X_feat.shape}')
print(f'y      shape : {y.shape}')

### 3.5 — Normalisation (fit on normal only)

In [ ]:
# only use normal windows to compute mean/std (unsupervised setup)
normal_mask = (y == 0)
print(f'Normal windows : {normal_mask.sum()}')
print(f'Fault  windows : {(~normal_mask).sum()}')

scaler_raw  = StandardScaler()
scaler_feat = StandardScaler()

scaler_raw.fit(X_raw[normal_mask])
scaler_feat.fit(X_feat[normal_mask])

X_raw_norm  = scaler_raw.transform(X_raw).astype(np.float32)
X_feat_norm = scaler_feat.transform(X_feat).astype(np.float32)

print(f'\nAfter normalisation (normal windows):')
print(f'  mean ≈ {X_raw_norm[normal_mask].mean():.4f}  (should be ~0)')
print(f'  std  ≈ {X_raw_norm[normal_mask].std():.4f}   (should be ~1)')

In [ ]:
# before vs after normalisation
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
axes[0].hist(X_raw[normal_mask].flatten(), bins=60, color='steelblue', alpha=0.7)
axes[0].set_title('Raw signal values — Normal windows')
axes[0].set_xlabel('Amplitude')

axes[1].hist(X_raw_norm[normal_mask].flatten(), bins=60, color='darkorange', alpha=0.7)
axes[1].set_title('After StandardScaler — Normal windows')
axes[1].set_xlabel('Normalised amplitude')

for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 3.6 — Save processed data

In [ ]:
# save scaler parameters for inference time use
np.save(os.path.join(OUT_DIR, 'cwru_scaler_mean.npy'),  scaler_raw.mean_)
np.save(os.path.join(OUT_DIR, 'cwru_scaler_scale.npy'), scaler_raw.scale_)

out_path = os.path.join(OUT_DIR, 'cwru_processed.npz')
np.savez_compressed(
    out_path,
    X_raw      = X_raw_norm,
    X_features = X_feat_norm,
    y_labels   = y,
    y_names    = y_names
)
print(f'Saved: {out_path}')
print(f'  X_raw      : {X_raw_norm.shape}')
print(f'  X_features : {X_feat_norm.shape}')
print(f'  File size  : {os.path.getsize(out_path)/1e6:.1f} MB')

### 3.7 — Verify saved file & class distribution

In [ ]:
# reload and verify
check = np.load(out_path, allow_pickle=True)
print('Keys:', list(check.keys()))
for k in check.keys():
    print(f'  {k}: {check[k].shape}')

In [ ]:
# window count per class
unique_names, counts = np.unique(y_names, return_counts=True)

plt.figure(figsize=(10, 4))
colors = ['#4CAF50' if n == 'Normal' else '#F44336' for n in unique_names]
bars   = plt.bar(unique_names, counts, color=colors)
for bar, c in zip(bars, counts):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
             str(c), ha='center', fontsize=8)
plt.xticks(rotation=30, ha='right', fontsize=8)
plt.ylabel('Number of Windows')
plt.title('Window Count per Class — CWRU')
plt.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# PCA to see if normal and fault clusters separate in 2D
from sklearn.decomposition import PCA

# use a sample for speed
sample_idx = np.random.choice(len(X_feat_norm), size=2000, replace=False)
X_sample   = X_feat_norm[sample_idx]
y_sample   = (y[sample_idx] > 0).astype(int)

pca    = PCA(n_components=2)
X_pca  = pca.fit_transform(X_sample)

plt.figure(figsize=(7, 5))
plt.scatter(X_pca[y_sample==0, 0], X_pca[y_sample==0, 1],
            s=5, alpha=0.5, color='green', label='Normal')
plt.scatter(X_pca[y_sample==1, 0], X_pca[y_sample==1, 1],
            s=5, alpha=0.5, color='red',   label='Fault')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.title('PCA — Feature Space (Normal vs Fault)')
plt.legend(markerscale=3)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Explained variance: PC1={pca.explained_variance_ratio_[0]*100:.1f}%  PC2={pca.explained_variance_ratio_[1]*100:.1f}%')